In [2]:
from pathlib import Path
import pandas as pd

# Robust project root (works even if you're in notebooks/)
cwd = Path().resolve()
project_root = next(p for p in [cwd] + list(cwd.parents) if (p / "data").exists())

path_input  = project_root / "data" / "raw"
path_output = project_root / "data" / "procesed"   # as you wrote it
path_output.mkdir(parents=True, exist_ok=True)

SAMPLE_PER_MONTH = 1_000_000
RANDOM_STATE = 42

print("project_root:", project_root)
print("path_input:", path_input)
print("path_output:", path_output)

project_root: C:\Users\leodo\Desktop\NYC-Taxi-Anomaly-Classification
path_input: C:\Users\leodo\Desktop\NYC-Taxi-Anomaly-Classification\data\raw
path_output: C:\Users\leodo\Desktop\NYC-Taxi-Anomaly-Classification\data\procesed


In [3]:
def find_month_files(input_dir: Path, year: int = 2019) -> list[Path]:
    files = sorted(input_dir.glob(f"yellow_tripdata_{year}-*.csv"))
    if not files:
        raise FileNotFoundError(f"No monthly files found in {input_dir} for year={year}")
    return files

files_2019 = find_month_files(path_input, 2019)
[f.name for f in files_2019][:5], len(files_2019)

(['yellow_tripdata_2019-01.csv',
  'yellow_tripdata_2019-02.csv',
  'yellow_tripdata_2019-03.csv',
  'yellow_tripdata_2019-04.csv',
  'yellow_tripdata_2019-05.csv'],
 12)

In [4]:
def get_union_columns(files: list[Path]) -> list[str]:
    union = set()
    for f in files:
        cols = pd.read_csv(f, nrows=0).columns.tolist()
        union.update(cols)
    # stable order: keep TLC-like ordering by sorting
    return sorted(union)

all_cols = get_union_columns(files_2019)
print("Total union columns:", len(all_cols))
all_cols

Total union columns: 18


['DOLocationID',
 'PULocationID',
 'RatecodeID',
 'VendorID',
 'congestion_surcharge',
 'extra',
 'fare_amount',
 'improvement_surcharge',
 'mta_tax',
 'passenger_count',
 'payment_type',
 'store_and_fwd_flag',
 'tip_amount',
 'tolls_amount',
 'total_amount',
 'tpep_dropoff_datetime',
 'tpep_pickup_datetime',
 'trip_distance']

In [5]:
def month_from_filename(file_path: Path) -> int:
    # yellow_tripdata_2019-01.csv -> 01
    return int(file_path.stem.split("_")[-1].split("-")[1])

def read_and_sample_month(file_path: Path, all_cols: list[str],
                          n: int = 1_000_000, random_state: int = 42) -> pd.DataFrame:
    df = pd.read_csv(file_path, low_memory=False)

    # sample (only if needed)
    if len(df) > n:
        df = df.sample(n=n, random_state=random_state)

    # align columns to union (adds missing columns as NA)
    df = df.reindex(columns=all_cols)

    # add metadata
    df["year"] = 2019
    df["month"] = month_from_filename(file_path)
    df["source_file"] = file_path.name

    return df

parts = []
for f in files_2019:
    df_m = read_and_sample_month(f, all_cols, SAMPLE_PER_MONTH, RANDOM_STATE)
    parts.append(df_m)
    print(f"{f.name}: sampled {len(df_m):,} rows | cols={df_m.shape[1]:,}")

master = pd.concat(parts, ignore_index=True)
print("MASTER SHAPE:", master.shape)
master.head()

yellow_tripdata_2019-01.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-02.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-03.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-04.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-05.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-06.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-07.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-08.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-09.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-10.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-11.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-12.csv: sampled 1,000,000 rows | cols=21
MASTER SHAPE: (12000000, 21)


,DOLocationID,PULocationID,RatecodeID,VendorID,congestion_surcharge,extra,fare_amount,improvement_surcharge,mta_tax,passenger_count,...,store_and_fwd_flag,tip_amount,tolls_amount,total_amount,tpep_dropoff_datetime,tpep_pickup_datetime,trip_distance,year,month,source_file
0,234,164,1.0,2.0,NaN,0.0,4.5,0.3,0.5,1.0,...,N,1.06,0.0,6.36,2019-01-09 09:31:43,2019-01-09 09:28:06,0.52,2019,1,yellow_tripdata_2019-01.csv
1,230,100,1.0,2.0,NaN,0.0,7.0,0.3,0.5,1.0,...,N,1.95,0.0,9.75,2019-01-02 07:37:09,2019-01-02 07:29:10,1.15,2019,1,yellow_tripdata_2019-01.csv
2,162,140,1.0,2.0,NaN,1.0,10.5,0.3,0.5,1.0,...,N,0.00,0.0,12.30,2019-01-07 16:06:42,2019-01-07 15:55:27,2.44,2019,1,yellow_tripdata_2019-01.csv
3,239,151,1.0,1.0,NaN,0.0,5.5,0.3,0.5,1.0,...,N,1.25,0.0,7.55,2019-01-09 06:56:05,2019-01-09 06:52:41,1.20,2019,1,yellow_tripdata_2019-01.csv
4,260,140,1.0,1.0,NaN,0.0,20.0,0.3,0.5,1.0,...,N,0.00,0.0,20.80,2019-01-17 09:14:37,2019-01-17 08:50:24,4.60,2019,1,yellow_tripdata_2019-01.csv


In [6]:
out_file = path_output / "master_2019_1M_per_month.parquet"
master.to_parquet(out_file, index=False)
print("Saved:", out_file)

Saved: C:\Users\leodo\Desktop\NYC-Taxi-Anomaly-Classification\data\procesed\master_2019_1M_per_month.parquet
